In [ ]:
# Install necessary libraries if they are not already in the Colab environment.
# markdownify is the key library for HTML to Markdown conversion.
!pip install requests beautifulsoup4 markdownify

import requests
from bs4 import BeautifulSoup
from markdownify import markdownify as md
from google.colab import files
import re

def scrape_to_markdown():
    """
    Prompts the user for a URL, scrapes the website's content, converts it
    to Markdown, and triggers a download for the resulting .md file.
    """
    print("Welcome to the Python Website to Markdown Scraper for Colab!")

    # 1. Get URL from the user
    url = input("Please enter the full URL of the website you want to scrape (e.g., https://example.com): ")

    if not url.startswith(('http://', 'https://')):
        url = 'https://' + url
        print(f"URL adjusted to: {url}")

    try:
        # 2. Fetch the website content
        print(f"\nScraping {url}...")
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        response = requests.get(url, headers=headers, timeout=15)
        # Raise an exception for bad status codes (4xx or 5xx)
        response.raise_for_status()
        print("Successfully fetched the website content.")

        # 3. Convert HTML to Markdown
        print("Converting HTML to Markdown...")
        # We use response.text which is the decoded text content
        html_content = response.text
        markdown_content = md(html_content, heading_style="ATX")
        print("Conversion complete.")

        # 4. Prepare the file for download
        # Create a clean filename from the URL
        domain = re.search(r'https?://(?:www\.)?([^/]+)', url).group(1)
        filename = f"{domain.replace('.', '_')}_content.md"

        # 5. Save the Markdown content to a file in the Colab environment
        with open(filename, 'w', encoding='utf-8') as f:
            f.write(markdown_content)
        print(f"Markdown content saved to '{filename}'.")

        # 6. Use Colab's files module to trigger a browser download
        print("\nTriggering file download...")
        files.download(filename)
        print(f"Download of '{filename}' should start shortly.")

    except requests.exceptions.RequestException as e:
        print(f"\n--- ERROR ---")
        print(f"An error occurred while trying to fetch the URL: {e}")
        print("Please check the URL and your internet connection.")
    except Exception as e:
        print(f"\n--- ERROR ---")
        print(f"An unexpected error occurred: {e}")

# --- Run the scraper ---
scrape_to_markdown()